# Plotting correlations

Almost every brain-map result ends up as a scatter plot with a coefficient on it.
{func}`~snaplab_tools.plotting.plotting.plot_correlation` draws that plot: scatter, fit line,
confidence band, and an annotated statistic, all sized for a journal figure rather than a screen.

This tutorial covers the options you will actually reach for — swapping the correlation method,
fitting and comparing polynomials, colouring points by brain system, and replacing the parametric
p-value with an empirical one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from snaplab_tools.datasets import make_spatial_map, make_correlated_map, schaefer_systems
from snaplab_tools.plotting.plotting import plot_correlation, plot_correlation_unity
from snaplab_tools.plotting.utils import set_plotting_params

set_plotting_params()

## Some data

Two spatially autocorrelated maps over the real Schaefer 400 geometry, built to correlate at 0.4.
Think of them as any two parcellated measurements — myelin and intrinsic timescale, cortical
thickness and gene expression, whatever you have.

In [ ]:
x = make_spatial_map(n_regions=400, seed=0)
y = make_correlated_map(x, rho=0.4, seed=1)

print(x.shape, y.shape)

## The basic plot

Pass the two vectors and an axis. The function draws into the axis you give it rather than making
its own figure, so it composes into multi-panel layouts.

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=150)
plot_correlation(x, y, ax, x_label='Map X', y_label='Map Y')
plt.show()

Pearson is the default. Spearman is often the better choice for brain maps, which are frequently
skewed and occasionally have a few extreme parcels dragging a Pearson coefficient around.

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=150)
plot_correlation(x, y, ax, x_label='Map X', y_label='Map Y', method='spearman')
plt.show()

## Is it actually linear?

`auto_polynomial=True` fits each polynomial order in `models_to_test` and keeps the one that
explains the most variance, so you find out whether a straight line was the right summary rather
than assuming it.

In [ ]:
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=150)
plot_correlation(x, y, ax, x_label='Map X', y_label='Map Y',
                 auto_polynomial=True, models_to_test=[1, 2, 3])
plt.show()

## Colouring by brain system

Pass `data_group` to colour points by a categorical label. Yeo system assignments come straight
from the bundled parcellation via {func}`~snaplab_tools.datasets.schaefer_systems`.

This is worth doing routinely: a correlation driven entirely by one system separating from the
rest is a different finding from one that holds within systems, and the plain scatter cannot tell
them apart.

In [ ]:
systems = schaefer_systems(n_regions=400)
print(np.unique(systems))

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 2.5), dpi=150)
plot_correlation(x, y, ax, x_label='Map X', y_label='Map Y',
                 method='spearman', data_group=systems)
plt.show()

## Replacing the p-value with an empirical null

The annotated p-value assumes independent observations. Parcels are not independent, so for a
brain-map correlation that p-value is optimistic — often dramatically so.

`custom_inset` takes a null distribution, uses it to compute the p-value, and embeds a small
histogram showing where the observed effect falls. The figure then carries its own evidence
instead of asking the reader to trust an asterisk.

The surrogates come from {func}`~snaplab_tools.nulls.generate_surrogates`, which preserves the
spatial autocorrelation of the map while scrambling its relationship to anything else. The method
is BrainSMASH — it matches the empirical variogram of the original map, so the surrogates share
its smoothness without sharing its alignment to anything you are testing against. See
{mod}`snaplab_tools.nulls` for how to choose a distance basis.

:::{admonition} Please cite
:class: seealso

Burt, J.B., Helmer, M., Shinn, M., Anticevic, A., & Murray, J.D. (2020). Generative modeling of
brain maps with spatial autocorrelation. *NeuroImage*, 220, 117038.
<https://doi.org/10.1016/j.neuroimage.2020.117038>
:::

In [ ]:
from snaplab_tools.nulls import generate_surrogates
import scipy.stats as st

surrogates = generate_surrogates(y, n_perms=500, seed=0)
null_distribution = np.array([st.spearmanr(x, s)[0] for s in surrogates])

print(f'observed rho = {st.spearmanr(x, y)[0]:.3f}')
print(f'null: mean {null_distribution.mean():.3f}, sd {null_distribution.std():.3f}')

The null has a standard deviation of about 0.16. A naive test assumes roughly 0.05 at this sample
size (1/sqrt(n-3) with n = 400), so the honest null is **three times wider** than the parametric
one. That gap is exactly the correction being applied.

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 2.5), dpi=150)
plot_correlation(x, y, ax, x_label='Map X', y_label='Map Y',
                 method='spearman', data_group=systems,
                 custom_inset={'custom_null': null_distribution})
plt.show()

## Unity plots

When X and Y are the *same* measurement under two conditions — rest versus task, session 1 versus
session 2, patients versus controls — a normal correlation plot answers the wrong question. It
tells you whether they covary, when what you want to know is whether one is systematically larger.

{func}`~snaplab_tools.plotting.plotting.plot_correlation_unity` puts the X=Y line on the diagonal,
so a systematic shift shows up as points sitting off it.

In [ ]:
rng = np.random.default_rng(0)
n_subjects = 120

condition_a = rng.normal(0.45, 0.12, n_subjects)
# Same subjects, second condition: correlated with the first, but shifted upward.
condition_b = 0.7 * condition_a + 0.3 * rng.normal(0.45, 0.12, n_subjects) + 0.06

fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=150)
plot_correlation_unity(condition_a, condition_b, ax,
                       x_label='Rest', y_label='Task',
                       show_correlation=True, correlation_type='spearman',
                       show_marginals=True, grid=False)
plt.show()

Most subjects sit above the diagonal: the task condition is systematically higher. A standard
correlation plot of the same data would have shown a strong positive relationship and said nothing
at all about the shift, because the shift is exactly what a correlation removes.

## See also

- {func}`~snaplab_tools.plotting.utils.set_plotting_params` and
  {func}`~snaplab_tools.plotting.utils.get_my_colors` — the style and palette behind these figures
- {mod}`snaplab_tools.nulls` — where the `custom_inset` null came from
- {func}`~snaplab_tools.stats.compute_stat` — the statistic these plots annotate